# ARMAS 5M versus 1H final-results comparison

Run the matched 5M and 1H final pipelines first. This notebook reads their saved artifacts, checks that comparable protocols were used, and creates report-ready cadence comparisons without retraining either model.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', context='notebook')
try:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/ARMAS_project_outputs')
except ImportError:
    ROOT = Path('outputs')

OUTPUT_DIRS = {
    '5M': ROOT / 'physics_guided_residual_5m',
    '1H': ROOT / 'physics_guided_residual_1h',
}
COMPARISON_DIR = ROOT / 'cadence_comparison_5m_vs_1h'
COMPARISON_DIR.mkdir(parents=True, exist_ok=True)
for cadence, folder in OUTPUT_DIRS.items():
    if not folder.exists():
        raise FileNotFoundError(f'Run the {cadence} notebook first; missing {folder}')
print('Comparison outputs:', COMPARISON_DIR)


## 1. Load matching artifacts

P2 remains the validation comparison. P3 is the final test and is shown when both runs have completed test evaluation. The six-assignment comparison is likewise optional until both full runs finish.


In [ ]:
def load_pair(filename, required=True):
    frames = []
    for cadence, folder in OUTPUT_DIRS.items():
        path = folder / filename
        if not path.exists():
            if required:
                raise FileNotFoundError(f'Missing {path}')
            return None
        frames.append(pd.read_csv(path).assign(cadence=cadence))
    return pd.concat(frames, ignore_index=True)

p2 = load_pair('p2_validation_results.csv')
p3 = load_pair('p3_locked_test_results.csv', required=False)
rotations = load_pair('all_six_partition_assignments_long.csv', required=False)
complexity = load_pair('network_complexity.csv', required=False)
print('P2 rows:', len(p2))
print('P3 available:', p3 is not None)
print('Six assignments available:', rotations is not None)
display(p2.head())


## 2. Validation comparison

Use P2 to compare cadences and choose configurations. Keep P3 locked until the choices for both cadence experiments are frozen.


In [ ]:
required = {'model', 'regime', 'RMSE', 'cadence'}
missing = required - set(p2.columns)
if missing:
    raise KeyError(f'P2 tables are missing columns: {sorted(missing)}')
p2_table = p2.pivot_table(index=['model', 'regime'], columns='cadence', values='RMSE')
if {'5M', '1H'}.issubset(p2_table.columns):
    p2_table['1H_minus_5M_RMSE'] = p2_table['1H'] - p2_table['5M']
display(p2_table.sort_values(['regime', '5M']).round(4))

plot = p2[p2.regime.isin(['all', 'A > 15'])].copy()
g = sns.catplot(data=plot, x='RMSE', y='model', hue='cadence', col='regime',
                kind='bar', sharex=False, height=5, aspect=1.05)
g.fig.suptitle('P2 validation: 5M versus 1H', y=1.03)
plt.show()
p2.to_csv(COMPARISON_DIR / 'p2_cadence_comparison_long.csv', index=False)
p2_table.to_csv(COMPARISON_DIR / 'p2_cadence_comparison_rmse.csv')


### Physics and architecture contrasts on validation data

These contrasts isolate three questions: whether temporal context beats the current-time MLP, whether learning a correction to NAIRAS beats direct prediction, and whether the longer 1H context beats the short 5M context. Positive gain means the first named method reduced RMSE.


In [ ]:
def rmse_lookup(frame, cadence, model, regime):
    match = frame[(frame.cadence == cadence) & (frame.model == model) & (frame.regime == regime)]
    return float(match.RMSE.iloc[0]) if len(match) else np.nan

contrast_rows = []
for cadence in ['5M', '1H']:
    for regime in ['all', 'A <= 15', 'A > 15']:
        mlp = rmse_lookup(p2, cadence, 'MLP', regime)
        cnn = rmse_lookup(p2, cadence, 'Hybrid CNN-MLP', regime)
        direct = rmse_lookup(p2, cadence, 'Direct Transformer', regime)
        physics = rmse_lookup(p2, cadence, 'Physics-guided Transformer', regime)
        nairas = rmse_lookup(p2, cadence, 'NAIRAS V3', regime)
        contrast_rows += [
            {'cadence': cadence, 'regime': regime, 'question': 'CNN temporal gain over MLP', 'RMSE_gain': mlp - cnn},
            {'cadence': cadence, 'regime': regime, 'question': 'Direct Transformer gain over MLP', 'RMSE_gain': mlp - direct},
            {'cadence': cadence, 'regime': regime, 'question': 'Physics-guided gain over direct Transformer', 'RMSE_gain': direct - physics},
            {'cadence': cadence, 'regime': regime, 'question': 'Physics-guided gain over NAIRAS V3', 'RMSE_gain': nairas - physics},
        ]
physics_contrasts = pd.DataFrame(contrast_rows)
display(physics_contrasts.pivot_table(index=['question', 'regime'], columns='cadence', values='RMSE_gain').round(4))
sns.catplot(data=physics_contrasts[physics_contrasts.regime.isin(['all', 'A > 15'])],
            x='RMSE_gain', y='question', hue='cadence', col='regime',
            kind='bar', sharex=False, height=5, aspect=1.1)
plt.axvline(0, color='black', linewidth=.8)
plt.show()
physics_contrasts.to_csv(COMPARISON_DIR / 'p2_physics_architecture_contrasts.csv', index=False)


### Feature-by-lag physics exploration

The saved P1 Spearman matrices describe association between each environmental input and `ARMAS − NAIRAS V3`. Comparing their strongest lags tests whether discrepancies are associated with fast particle/solar-wind variation or longer geomagnetic and solar context. This is exploratory association, not causal evidence.


In [ ]:
lag_frames = []
for cadence, folder in OUTPUT_DIRS.items():
    path = folder / 'p1_residual_feature_by_lag_correlations.csv'
    if not path.exists():
        raise FileNotFoundError(f'Missing {path}; finish the exploration section in both notebooks.')
    wide = pd.read_csv(path, index_col=0)
    long = wide.rename_axis('feature').reset_index().melt(
        id_vars='feature', var_name='lag', value_name='Spearman')
    long['cadence'] = cadence
    long['absolute_Spearman'] = long.Spearman.abs()
    lag_frames.append(long)
lag_comparison = pd.concat(lag_frames, ignore_index=True)
peak_lags = (lag_comparison.sort_values('absolute_Spearman', ascending=False)
             .groupby(['cadence', 'feature'], as_index=False).first())
display(peak_lags.sort_values(['cadence', 'absolute_Spearman'], ascending=[True, False])
        .groupby('cadence').head(15).round(4))
top_features = (peak_lags.groupby('feature').absolute_Spearman.max()
                .nlargest(15).index)
sns.catplot(data=peak_lags[peak_lags.feature.isin(top_features)],
            x='absolute_Spearman', y='feature', hue='cadence', kind='bar',
            height=7, aspect=1.25)
plt.show()
lag_comparison.to_csv(COMPARISON_DIR / 'p1_feature_lag_correlations_by_cadence.csv', index=False)
peak_lags.to_csv(COMPARISON_DIR / 'p1_peak_feature_lags_by_cadence.csv', index=False)


## 3. Locked P3 and radiation-regime comparison

This section runs only after both cadence notebooks have evaluated P3. Report overall and high-dose results together; a cadence can improve average error while worsening `ARMAS > 15`.


In [ ]:
if p3 is None:
    print('P3 comparison is locked or incomplete. Finish both cadence runs first.')
else:
    p3_table = p3.pivot_table(index=['model', 'regime'], columns='cadence', values='RMSE')
    p3_table['1H_minus_5M_RMSE'] = p3_table['1H'] - p3_table['5M']
    display(p3_table.round(4))
    sns.catplot(data=p3[p3.regime.isin(['all', 'A <= 15', 'A > 15'])],
                x='RMSE', y='model', hue='cadence', col='regime',
                kind='bar', sharex=False, height=5, aspect=.9)
    plt.show()
    p3.to_csv(COMPARISON_DIR / 'p3_cadence_comparison_long.csv', index=False)
    p3_table.to_csv(COMPARISON_DIR / 'p3_cadence_comparison_rmse.csv')


### Paired cadence uncertainty on identical P3 rows

When both prediction tables exist, this paired bootstrap compares 1H and 5M errors on the same ARMAS observations. A negative `1H_minus_5M_RMSE` favors 1H; a positive value favors 5M. Because rows within flights can be correlated, use partition rotations as the stronger robustness result.


In [ ]:
prediction_paths = {c: folder / 'p3_predictions_by_row.csv' for c, folder in OUTPUT_DIRS.items()}
if not all(path.exists() for path in prediction_paths.values()):
    print('Paired P3 comparison remains locked until both prediction tables exist.')
else:
    pred_5m = pd.read_csv(prediction_paths['5M']).sort_values('row_index').reset_index(drop=True)
    pred_1h = pd.read_csv(prediction_paths['1H']).sort_values('row_index').reset_index(drop=True)
    if len(pred_5m) != len(pred_1h) or not np.allclose(pred_5m.ARMAS, pred_1h.ARMAS):
        raise ValueError('The cadence prediction tables do not contain identical ordered ARMAS targets.')
    common_models = sorted((set(pred_5m.columns) & set(pred_1h.columns)) - {'row_index', 'ARMAS', 'NAIRAS V3'})
    y = pred_5m.ARMAS.to_numpy()
    rng = np.random.default_rng(42)
    paired_rows = []
    for model in common_models:
        p5, p1 = pred_5m[model].to_numpy(), pred_1h[model].to_numpy()
        for regime, mask in {'all': np.ones(len(y), bool), 'A <= 15': y <= 15, 'A > 15': y > 15}.items():
            yy, a, b = y[mask], p5[mask], p1[mask]
            differences = []
            for _ in range(1000):
                index = rng.integers(0, len(yy), len(yy))
                rmse_1h = np.sqrt(np.mean((b[index] - yy[index]) ** 2))
                rmse_5m = np.sqrt(np.mean((a[index] - yy[index]) ** 2))
                differences.append(rmse_1h - rmse_5m)
            low, median, high = np.quantile(differences, [.025, .5, .975])
            paired_rows.append({'model': model, 'regime': regime,
                '1H_minus_5M_RMSE_median': median, 'CI_2.5%': low, 'CI_97.5%': high})
    paired_cadence_bootstrap = pd.DataFrame(paired_rows)
    display(paired_cadence_bootstrap.round(4))
    paired_cadence_bootstrap.to_csv(COMPARISON_DIR / 'p3_paired_1h_vs_5m_bootstrap.csv', index=False)


## 4. Partition robustness and efficiency


In [ ]:
if rotations is not None:
    metric_columns = [c for c in ['RMSE', 'MAE', 'R2', 'bias'] if c in rotations.columns]
    grouping = [c for c in ['cadence', 'model', 'regime', 'split'] if c in rotations.columns]
    rotation_summary = rotations.groupby(grouping)[metric_columns].agg(['mean', 'std'])
    display(rotation_summary.round(4))
    rotations.to_csv(COMPARISON_DIR / 'all_assignments_cadence_comparison.csv', index=False)
else:
    print('Run all six assignments in both notebooks to compare partition robustness.')

if complexity is not None:
    keep = [c for c in ['cadence', 'model', 'trainable_parameters',
          'parameter_and_buffer_MB', 'approx_FLOPs_per_observation',
          'latency_ms_per_observation'] if c in complexity.columns]
    display(complexity[keep].sort_values(['model', 'cadence']).round(4))
    complexity.to_csv(COMPARISON_DIR / 'network_complexity_by_cadence.csv', index=False)


## 5. Cross-cadence attribution consistency

On the P3 final test, compare integrated-gradient strength by physical variable group. Agreement across cadences strengthens an association claim; disagreement can indicate scale sensitivity or correlated predictors sharing attribution.


In [ ]:
ig_paths = {c: folder / 'p3_integrated_gradients_variable_lag.csv' for c, folder in OUTPUT_DIRS.items()}
if not all(path.exists() for path in ig_paths.values()):
    print('Integrated-gradient comparison is unavailable until both P3 attribution sections run.')
else:
    def physics_group(feature):
        if feature.startswith('NM_'): return 'neutron monitors'
        if feature.startswith('SXR_'): return 'solar X-ray'
        if feature.startswith('Particles_'): return 'energetic particles'
        if feature.startswith('SW_'): return 'solar wind'
        if feature.startswith('Index_'): return 'geomagnetic indices'
        if feature.startswith('Solar_'): return 'solar activity'
        return 'other'
    ig_frames = []
    for cadence, path in ig_paths.items():
        frame = pd.read_csv(path)
        frame['cadence'] = cadence
        frame['physics_group'] = frame.feature.map(physics_group)
        ig_frames.append(frame)
    ig = pd.concat(ig_frames, ignore_index=True)
    ig_group = (ig.groupby(['cadence', 'cohort', 'physics_group'], as_index=False)
                .mean(numeric_only=True))
    display(ig_group.sort_values(['cohort', 'cadence', 'mean_absolute_IG'],
                                  ascending=[True, True, False]).round(6))
    sns.catplot(data=ig_group, x='mean_absolute_IG', y='physics_group',
                hue='cadence', col='cohort', kind='bar', sharex=False,
                height=5, aspect=1.05)
    plt.show()
    ig.to_csv(COMPARISON_DIR / 'integrated_gradients_by_cadence.csv', index=False)
    ig_group.to_csv(COMPARISON_DIR / 'integrated_gradients_physics_groups.csv', index=False)


## 6. Report questions

1. On P2, which cadence has lower overall RMSE for each architecture?
2. Does that conclusion hold for `ARMAS > 15`?
3. After configurations are frozen, does the same cadence win on locked P3?
4. Are cadence differences stable across all six partition assignments?
5. Does the 24-hour 1H context improve results enough to justify its compute and latency relative to the one-hour 5M context?
6. Do attention and integrated-gradient maps identify consistent variable groups at the two time scales?

Treat attribution as association. State the cadence, temporal span, run mode, selected loss, and partition role beside every reported result.
